In [ ]:
!pip install pypdf

In [ ]:
!pip install chromadb
!pip install streamlit pyngrok

In [ ]:
pip install langchain-huggingface langchain-chroma

  Using cached langchain_core-1.2.19-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.2.19-py3-none-any.whl (503 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.28 requires langchain-core<1.0.0,>=0.3.73, but you have langchain-core 1.2.19 which is incompatible.
langchain-community 0.3.27 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.2.19 which is incompatible.


In [ ]:
# Step 1: Upgrade packages that conflict with Colab's pre-installed environment
!pip install -q -U langchain-core langchain-text-splitters google-ai-generativelanguage

# Step 2: Install project dependencies
!pip install -q -U PyPDF2 langchain langchain-community langchain-google-genai \
    sentence-transformers faiss-cpu nest-asyncio "requests==2.32.4"

from IPython.display import display, Javascript
display(Javascript("google.colab.kernel.restartRuntime()"))


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.28 requires langchain-core<1.0.0,>=0.3.73, but you have langchain-core 1.2.19 which is incompatible.
langchain 0.3.28 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.1.1 which is incompatible.
langchain-community 0.3.27 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.2.19 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.


<IPython.core.display.Javascript object>

In [ ]:
import streamlit as st
import requests
import time
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
#google api
from langchain_google_genai import ChatGoogleGenerativeAI#Google API Key

os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"


In [ ]:
# Download Required PDFs

def download_pdfs():

    urls = [
        "https://arxiv.org/pdf/2506.02153",
        "https://assets.anthropic.com/m/71876fabef0f0ed4/original/reasoning_models_paper.pdf"
    ]

    pdf_files = []

    for i, url in enumerate(urls):

        filename = f"paper{i}.pdf"

        if not os.path.exists(filename):

            response = requests.get(url)

            with open(filename, "wb") as f:
                f.write(response.content)

        pdf_files.append(filename)

    return pdf_files

In [ ]:
# Load PDFs
def load_documents(pdf_paths):

    docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(path)
        docs.extend(loader.load())

    return docs

In [ ]:
# Split Documents

def split_documents(documents):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

    return chunks

In [ ]:
# Create Vector Database

def create_vector_db(chunks):

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="chroma_db"
    )

    vectordb.persist()

    return vectordb


In [ ]:
# Load Vector Database

def load_vector_db():

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectordb = Chroma(
        persist_directory="chroma_db",
        embedding_function=embeddings
    )

    return vectordb


In [ ]:
# Build QA Chain

def build_qa_chain(vectordb):

    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        temperature=0.3
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectordb.as_retriever(search_kwargs={"k":3}),
        return_source_documents=True
    )

    return qa_chain


In [ ]:
# Streamlit UI

st.title("Searchable PDF Q&A System")

st.write("Ask questions about the provided research papers.")


if not os.path.exists("chroma_db"):

    with st.spinner("Downloading and processing PDFs..."):

        pdfs = download_pdfs()

        documents = load_documents(pdfs)

        chunks = split_documents(documents)

        create_vector_db(chunks)

    st.success("PDFs processed and vector database created!")


vectordb = load_vector_db()

qa_chain = build_qa_chain(vectordb)


query = st.text_input("Enter your question:")


if query:

    start_time = time.time()

    result = qa_chain.invoke({"query": query})

    end_time = time.time()
    st.subheader("Answer")

    st.write(result["result"])


    st.subheader("Context Used")

    for doc in result["source_documents"]:
        st.write(doc.page_content)
        st.write("---")


    st.write("Response Time:", round(end_time - start_time, 2), "seconds")

2026-03-15 06:35:10.148 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:10.261 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-15 06:35:10.263 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:10.264 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:10.265 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:10.266 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:10.267 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:10.268 Thread 'MainThread': mi

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_4866/3194922400.py:15: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
2026-03-15 06:35:35.713 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:35.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:35.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_4866/2404925166.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(
2026-03-15 06:35:37.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:37.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 06:35:37.240 Thread 'Mai